# Phase 4 — Model Comparison

[`walkthrough/phase4.md`](../walkthrough/phase4.md). After the Phase 1 leakage check the headline feature
set is the deployable one: income, demographics, debt and the four categoricals. The full set
(adding the 11 spending shares and 5 participation indicators) is run alongside as a
diagnostic upper bound, because with household size and income the shares reconstruct total
spend. Everything below reads the training split under PSU-grouped CV; the held-out split is
touched once, in the final cell.

In [1]:
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier

from savings_goal import config as C
from savings_goal import viz
from savings_goal.evaluation.cv import cross_validate_grouped, grouped_cv, oof_predict_proba
from savings_goal.evaluation.metrics import fold_metrics, probability_metrics
from savings_goal.features.engineer import spec_from_frame
from savings_goal.io import load_features, write_result
from savings_goal.models.pipeline import (
    XGB_DEFAULT,
    XGB_SEARCH_SPACE,
    logistic,
    model_zoo,
    preprocessor,
    save_model_final,
    xgb_pipeline,
)

viz.use()
feats = load_features()
spec = spec_from_frame(feats)
train = feats[~feats["Is_Test"]].reset_index(drop=True)
test = feats[feats["Is_Test"]].reset_index(drop=True)
y, groups = train["Goal_Met"], train[C.GROUP_COL]
FEATURE_SETS = {"deployable": spec.deployable_numeric, "full": spec.numeric}
HEADLINE = "deployable"
print(f"train: {len(train):,} ({groups.nunique():,} PSUs)   held-out test: {len(test):,} "
      f"({test[C.GROUP_COL].nunique():,} PSUs, disjoint)")
print(f"Goal_Met (positive class = on track) prevalence: {y.mean():.3f}; "
      f"at-risk prevalence {1 - y.mean():.3f}")

EXTRA = {
    "precision (on-track)": lambda yt, s, yp: precision_score(yt, yp, zero_division=0),
    "recall (on-track)": lambda yt, s, yp: recall_score(yt, yp),
}

train: 33,219 (1,989 PSUs)   held-out test: 8,299 (472 PSUs, disjoint)
Goal_Met (positive class = on track) prevalence: 0.319; at-risk prevalence 0.681


## Q1/Q2 — Seven families under PSU-grouped stratified 5-fold CV, ranked on threshold-free metrics too

In [2]:
rows = []
for set_name, num in FEATURE_SETS.items():
    cols = num + C.CATEGORICALS
    zoo = model_zoo(num)
    zoo["Single income threshold"] = make_pipeline(SimpleImputer(strategy="median"),
                                                   DecisionTreeClassifier(max_depth=1, random_state=C.SEED))
    for name, model in zoo.items():
        use = ["Log_Income"] if name == "Single income threshold" else cols
        proba = name != "Linear SVM"
        folds = cross_validate_grouped(model, train[use], y, groups,
                                       {**fold_metrics(with_proba=proba), **EXTRA}, grouped_cv())
        row = {"feature set": set_name, "model": name, **folds.mean().to_dict(),
               "F1 sd": folds["F1 (macro)"].std(), "ROC-AUC sd": folds["ROC-AUC"].std()}
        if not proba:  # LinearSVC has no probabilities: Brier / log-loss / ECE are undefined
            row.update({"Brier": np.nan, "Log-loss": np.nan, "ECE": np.nan})
        rows.append(row)
    print(f"  done: {set_name}")
comparison = pd.DataFrame(rows)
show = ["accuracy", "F1 (macro)", "F1 sd", "ROC-AUC", "PR-AUC (at-risk)", "Brier", "Log-loss", "ECE"]
for set_name in FEATURE_SETS:
    sub = comparison[comparison["feature set"] == set_name].set_index("model")
    print(f"\n=== {set_name} features, ranked by ROC-AUC ===")
    print(sub[show].sort_values("ROC-AUC", ascending=False).round(4).to_string())

  done: deployable


  done: full

=== deployable features, ranked by ROC-AUC ===
                         accuracy  F1 (macro)   F1 sd  ROC-AUC  PR-AUC (at-risk)   Brier  Log-loss     ECE
model                                                                                                     
HistGradientBoosting       0.8112      0.7776  0.0065   0.8799            0.9406  0.1295    0.3992  0.0119
Logistic Regression        0.7911      0.7731  0.0068   0.8763            0.9378  0.1441    0.4446  0.1028
Linear SVM                 0.7914      0.7736  0.0068   0.8762            0.9378     NaN       NaN     NaN
XGBoost (untuned)          0.8092      0.7755  0.0051   0.8754            0.9382  0.1324    0.4094  0.0222
Random Forest              0.7984      0.7774  0.0052   0.8703            0.9337  0.1421    0.4419  0.0823
Decision Tree              0.7681      0.7506  0.0087   0.8499            0.9135  0.1587    0.6433  0.1038
Single income threshold    0.7781      0.7417  0.0040   0.7392            0.8077  0

## Q3 — Hyperparameter search (XGBoost on both feature sets; logistic C on the headline set)

In [3]:
searches, sensitivity = {}, {}
for set_name, num in FEATURE_SETS.items():
    search = RandomizedSearchCV(
        xgb_pipeline(num), {f"xgbclassifier__{k}": v for k, v in XGB_SEARCH_SPACE.items()},
        n_iter=15, cv=grouped_cv(), scoring={"f1_macro": "f1_macro", "roc_auc": "roc_auc",
                                             "neg_log_loss": "neg_log_loss"},
        refit="f1_macro", random_state=C.SEED, n_jobs=1,
    )
    search.fit(train[num + C.CATEGORICALS], y, groups=groups)
    searches[set_name] = search
    res = pd.DataFrame(search.cv_results_)
    sensitivity[set_name] = {k: float(res.groupby(f"param_xgbclassifier__{k}")["mean_test_f1_macro"].mean().std())
                             for k in XGB_SEARCH_SPACE}
    best = {k.split("__")[1]: v for k, v in search.best_params_.items()}
    i = search.best_index_
    print(f"{set_name}: best macro-F1 {search.best_score_:.4f}  ROC-AUC {res.loc[i, 'mean_test_roc_auc']:.4f}  "
          f"params {best}")
    print("  parameter sensitivity (sd of mean macro-F1 across levels):",
          {k: round(v, 5) for k, v in sorted(sensitivity[set_name].items(), key=lambda kv: -kv[1])})

lr_search = RandomizedSearchCV(
    make_pipeline(preprocessor(FEATURE_SETS[HEADLINE], linear=True), logistic()),
    {"logisticregression__C": [0.01, 0.1, 1.0, 10.0, 100.0]}, n_iter=5, cv=grouped_cv(),
    scoring="f1_macro", random_state=C.SEED, n_jobs=1,
)
lr_search.fit(train[FEATURE_SETS[HEADLINE] + C.CATEGORICALS], y, groups=groups)
print(f"\nlogistic regression ({HEADLINE}): best macro-F1 {lr_search.best_score_:.4f} "
      f"(C={lr_search.best_params_['logisticregression__C']})")

deployable: best macro-F1 0.7808  ROC-AUC 0.8822  params {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.08}
  parameter sensitivity (sd of mean macro-F1 across levels): {'max_depth': 0.00817, 'n_estimators': 0.00422, 'subsample': 0.00207, 'learning_rate': 0.00108, 'min_child_weight': 0.00088}


full: best macro-F1 0.8380  ROC-AUC 0.9292  params {'subsample': 0.7, 'n_estimators': 700, 'min_child_weight': 20, 'max_depth': 4, 'learning_rate': 0.08}
  parameter sensitivity (sd of mean macro-F1 across levels): {'learning_rate': 0.00431, 'subsample': 0.00147, 'n_estimators': 0.00122, 'max_depth': 0.00096, 'min_child_weight': 0.00063}



logistic regression (deployable): best macro-F1 0.7738 (C=0.01)


In [4]:
# Score the tuned configurations with exactly the same grouped folds and metrics as Table 1,
# so the table, the text and the held-out evaluation all describe the same model.
final = {"headline": HEADLINE, "untuned_params": XGB_DEFAULT, "search_space": XGB_SEARCH_SPACE,
         "cv": "StratifiedGroupKFold(5, shuffle=True, random_state=42) grouped by IDPSU",
         "split": "training split (Is_Test == False)"}
for set_name, num in FEATURE_SETS.items():
    search = searches[set_name]
    params = {k.split("__")[1]: v for k, v in search.best_params_.items()}
    folds = cross_validate_grouped(xgb_pipeline(num, params=params), train[num + C.CATEGORICALS], y, groups,
                                   {**fold_metrics(), **EXTRA}, grouped_cv())
    comparison = pd.concat([comparison, pd.DataFrame([{
        "feature set": set_name, "model": "XGBoost (tuned)", **folds.mean().to_dict(),
        "F1 sd": folds["F1 (macro)"].std(), "ROC-AUC sd": folds["ROC-AUC"].std()}])], ignore_index=True)
    final[set_name] = {"best_params": params, "best_score": float(search.best_score_),
                       "best_score_metric": "f1_macro",
                       "cv_metrics": folds.mean().to_dict(),
                       "parameter_sensitivity": sensitivity[set_name],
                       "n_features": len(num) + len(C.CATEGORICALS)}
save_model_final(final)
comparison.to_csv(C.RESULTS / "model_comparison.csv", index=False)

tbl = comparison.pivot(index="model", columns="feature set", values=["ROC-AUC", "F1 (macro)"])
print(tbl.sort_values(("ROC-AUC", HEADLINE), ascending=False).round(4).to_string())
gain = (tbl.loc["XGBoost (tuned)", ("F1 (macro)", HEADLINE)]
        - tbl.loc["XGBoost (untuned)", ("F1 (macro)", HEADLINE)])
print(f"\ntuning gain on the headline set: {gain:+.4f} macro-F1")
print("wrote results/model_final.json and results/model_comparison.csv")

                           ROC-AUC         F1 (macro)        
feature set             deployable    full deployable    full
model                                                        
XGBoost (tuned)             0.8817  0.9287     0.7797  0.8354
HistGradientBoosting        0.8799  0.9282     0.7776  0.8353
Logistic Regression         0.8763  0.9200     0.7731  0.8193
Linear SVM                  0.8762  0.9199     0.7736  0.8175
XGBoost (untuned)           0.8754  0.9293     0.7755  0.8353
Random Forest               0.8703  0.9148     0.7774  0.8229
Decision Tree               0.8499  0.8821     0.7506  0.7838
Single income threshold     0.7392  0.7392     0.7417  0.7417
Majority baseline           0.5000  0.5000     0.4050  0.4050

tuning gain on the headline set: +0.0042 macro-F1
wrote results/model_final.json and results/model_comparison.csv


## Q4 — Precision or recall? The at-risk view, threshold tuned on out-of-fold training scores

In [5]:
num = FEATURE_SETS[HEADLINE]
best = xgb_pipeline(num, params=final[HEADLINE]["best_params"])
oof = oof_predict_proba(best, train[num + C.CATEGORICALS], y, groups)
at_risk_true, at_risk_score = 1 - y.to_numpy(), 1 - oof
prec, rec, thr = precision_recall_curve(at_risk_true, at_risk_score)
f1 = 2 * prec * rec / (prec + rec + 1e-12)
i_best = int(np.argmax(f1[:-1]))
tuned_threshold = float(thr[i_best])
print(f"At-risk class (positive = Goal_Met 0, prevalence {at_risk_true.mean():.3f}), out-of-fold on train:")
print(f"  F1-optimal at-risk threshold {tuned_threshold:.3f}: precision {prec[i_best]:.3f}  "
      f"recall {rec[i_best]:.3f}  F1 {f1[i_best]:.3f}")
recall_targets = {}
for target in (0.80, 0.90, 0.95):
    # highest threshold whose recall still reaches the target (recall falls as the threshold rises)
    ok = np.flatnonzero(rec[:-1] >= target)
    j = int(ok[-1])
    recall_targets[target] = {"threshold": float(thr[j]), "precision": float(prec[j]), "recall": float(rec[j])}
    print(f"  recall >= {target:.0%}: precision {prec[j]:.3f} at recall {rec[j]:.3f} (threshold {thr[j]:.3f})")

At-risk class (positive = Goal_Met 0, prevalence 0.681), out-of-fold on train:
  F1-optimal at-risk threshold 0.437: precision 0.830  recall 0.911  F1 0.869
  recall >= 80%: precision 0.893 at recall 0.800 (threshold 0.654)
  recall >= 90%: precision 0.837 at recall 0.900 (threshold 0.463)
  recall >= 95%: precision 0.791 at recall 0.950 (threshold 0.318)


## Final evaluation on the held-out PSUs (touched once)

In [6]:
held_out = {}
for set_name, num in FEATURE_SETS.items():
    cols = num + C.CATEGORICALS
    model = xgb_pipeline(num, params=final[set_name]["best_params"]).fit(train[cols], y)
    p = model.predict_proba(test[cols])[:, 1]
    pred = (p >= 0.5).astype(int)
    held_out[set_name] = {**probability_metrics(test["Goal_Met"], p),
                          "F1 (macro) @0.5": float(classification_report(test["Goal_Met"], pred,
                                                   output_dict=True)["macro avg"]["f1-score"]),
                          "accuracy @0.5": float((pred == test["Goal_Met"]).mean())}
    if set_name == HEADLINE:
        print(f"FINAL — tuned XGBoost, {set_name} features, {len(test):,} held-out households\n")
        print(classification_report(test["Goal_Met"], pred, target_names=["at risk", "on track"], digits=3))
        flag = (1 - p) >= tuned_threshold
        tp = int((flag & (test["Goal_Met"] == 0)).sum())
        held_out[set_name]["tuned at-risk threshold"] = {
            "threshold": tuned_threshold, "precision": tp / flag.sum(),
            "recall": tp / int((test["Goal_Met"] == 0).sum()), "flagged": int(flag.sum())}
        print("Confusion matrix at 0.5 (rows = true):")
        print(pd.DataFrame(confusion_matrix(test["Goal_Met"], pred), index=["at risk", "on track"],
                           columns=["pred at risk", "pred on track"]).to_string())
        t = held_out[set_name]["tuned at-risk threshold"]
        print(f"\nat the at-risk threshold tuned on train ({tuned_threshold:.3f}): "
              f"precision {t['precision']:.3f}, recall {t['recall']:.3f} on held-out")
print("\nheld-out threshold-free metrics:")
print(pd.DataFrame(held_out).T.drop(columns=["tuned at-risk threshold"], errors="ignore").astype(float).round(4).to_string())
write_result("model_test", {"held_out": held_out, "recall_targets_oof": recall_targets,
                            "tuned_at_risk_threshold": tuned_threshold, "n_test": len(test),
                            "n_test_psu": int(test[C.GROUP_COL].nunique())})
print("wrote results/model_test.json")

FINAL — tuned XGBoost, deployable features, 8,299 held-out households

              precision    recall  f1-score   support

     at risk      0.850     0.881     0.865      5648
    on track      0.726     0.669     0.696      2651

    accuracy                          0.813      8299
   macro avg      0.788     0.775     0.781      8299
weighted avg      0.810     0.813     0.811      8299

Confusion matrix at 0.5 (rows = true):
          pred at risk  pred on track
at risk           4977            671
on track           877           1774

at the at-risk threshold tuned on train (0.437): precision 0.832, recall 0.912 on held-out



held-out threshold-free metrics:
            ROC-AUC  PR-AUC (on-track)  PR-AUC (at-risk)   Brier  Log-loss     ECE  F1 (macro) @0.5  accuracy @0.5
deployable    0.886             0.7750            0.9444  0.1266    0.3902  0.0082           0.7808         0.8135
full          0.931             0.8728            0.9664  0.0977    0.3074  0.0098           0.8370         0.8608
wrote results/model_test.json


In [7]:
plot = (comparison.pivot(index="model", columns="feature set", values="ROC-AUC")
        .sort_values(HEADLINE, ascending=False))
fig, ax = plt.subplots(figsize=(9, 5.5))
ys = np.arange(len(plot))[::-1]
for y, (model, row) in zip(ys, plot.iterrows()):
    ax.plot([row[HEADLINE], row["full"]], [y, y], color=viz.CONTEXT, lw=2, zorder=2)
viz.dot(ax, plot[HEADLINE], ys, viz.BLUE, size=8, zorder=3, label=f"{HEADLINE} (headline)")
viz.dot(ax, plot["full"], ys, viz.ORANGE, size=8, zorder=3, label="full (adds spending shares)")
ax.set_yticks(ys, plot.index)
ax.set_xlim(0.45, 1.0)
viz.strip(ax, "x")
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.set_xlabel("ROC-AUC, PSU-grouped 5-fold CV on the training split")
viz.legend(ax, loc="upper left")
ax.set_title(f"Adding spending shares lifts tuned XGBoost from {plot.loc['XGBoost (tuned)', HEADLINE]:.3f} "
             f"to {plot.loc['XGBoost (tuned)', 'full']:.3f}")
plt.savefig(C.RESULTS / "model_comparison.png")
plt.show()